# Week 1 — Digital Rock 데이터 탐색 (1조 Advanced)

## 이번 주 학습 목표
1. 3D voxel 데이터를 binary 파일에서 불러오고 세 축으로 시각화한다.
2. 세 개의 서로 다른 사암(BB, CastleGate, Bentheimer) 데이터의 통계를 비교한다.
3. **공극률(porosity, φ)** 의 정의와 "방향별 등방성(isotropy)" 개념을 정량적으로 검토한다.
4. **Sparse imaging** 시나리오를 직접 시뮬레이션하고, 가장 단순한 **선형 보간(linear interpolation)** 으로 누락 슬라이스를 복원한다.
5. **sparse 정도 k와 복원 오차의 관계** 를 측정하여, W2 이후 더 정교한 방법(deep learning)의 동기를 이해한다.

## 학습 방식
본 코스의 핵심은 **"직접 함수를 짜기보다, 배포된 코드의 파라미터를 바꿔보며 결과 변화를 관찰하는 것"** 입니다.
- 각 섹션에 **[Try-it!]** 박스가 있습니다. 거기서 변수 값을 다양하게 바꿔보세요.
- **[해석 질문]** 박스에서는 결과를 자신의 말로 설명하는 연습을 합니다.
- 마지막 **탐구 과제** 는 노트북 안에서 직접 코드를 수정하면서 답해주세요.

## 0. 환경 준비

**시작 전 확인**:
- 가상환경 `rock` 이 활성화되어 있는지 (`conda activate rock`)
- 이 notebook은 `group1_advanced/week1/notebooks/` 안에 있다는 가정으로 경로가 설정됨.

필요 패키지: `numpy`, `matplotlib`. (`COMMON/requirements_student.txt` 참조)

In [ ]:
import sys
from pathlib import Path

# helpers 모듈 import 경로 추가
sys.path.insert(0, str(Path('..').resolve() / 'helpers'))

import numpy as np
import matplotlib.pyplot as plt

from dr_utils import (
    load_volume, porosity, porosity_profile,
    show_three_axis, make_sparse,
    linear_interpolate_slice, reconstruct_sparse_linear, porosity_error,
    setup_plot_style, ORANGE, NAVY, GREEN, RED, GRAY,
)

setup_plot_style()
print('환경 준비 완료')

## 1. 데이터 로드 — 세 도메인 동시

본 W1에서 다룰 데이터:
| 도메인 | 출처 | shape | voxel 크기 | dtype | 값 |
|---|---|---|---|---|---|
| **BB** | Brazil sandstone | 256³ | 2.25 μm | uint8 | 0=solid, 1=pore |
| **CastleGate** | Castle Gate sandstone | 256³ | 2.25 μm | uint8 | 0/1 |
| **Bentheimer** | Bentheimer sandstone | 256³ | 2.25 μm | uint8 | 0/1 |

세 도메인 모두 "이미 segmentation(이진화)이 끝난" binary 부피입니다. (grayscale 단계는 W4에서 다룸)

In [ ]:
DATA_DIR = Path('..') / 'data'

domains = {
    'BB': load_volume(DATA_DIR / 'BB_256.bin'),
    'CastleGate': load_volume(DATA_DIR / 'CastleGate_256.bin'),
    'Bentheimer': load_volume(DATA_DIR / 'Bentheimer_256.bin'),
}

print(f"{'Domain':<15} {'shape':<18} {'dtype':<8} {'φ (%)':>8}")
print('-' * 55)
for name, vol in domains.items():
    print(f"{name:<15} {str(vol.shape):<18} {str(vol.dtype):<8} {porosity(vol)*100:>7.2f}")

> **[해석 질문 1]** 세 사암의 공극률이 다릅니다. 어느 사암이 "빈 공간이 많아" 액체/기체를 더 많이 저장할 수 있을까요?
> 이 공극률 값 자체로 "어느 사암이 더 \"좋은\" 저장소인가" 를 판단하기에 충분할까요? (힌트: 연결성도 중요)

## 2. 세 축 슬라이스 시각화

3D 부피는 한 번에 보기 어려우므로 단면(slice)으로 잘라봅니다.

- z축 slice: `volume[z, :, :]` → XY 평면 (micro-CT 측정 방향)
- y축 slice: `volume[:, y, :]` → ZX 평면
- x축 slice: `volume[:, :, x]` → ZY 평면

In [ ]:
# 세 도메인 모두 중앙 슬라이스 출력
for name, vol in domains.items():
    show_three_axis(vol, title_prefix=f'{name}:')
    plt.show()

> **[Try-it! ①]** 위 셀에서 `show_three_axis(vol, z=64, y=200, x=10, ...)` 처럼 인자를 추가해보세요.
> z, y, x 를 0, 64, 128, 192, 255 등으로 바꾸면 슬라이스가 어떻게 달라지는지 관찰해보세요.
>
> **[해석 질문 2]** 세 도메인 중 어느 사암이 가장 "균질(homogeneous)" 해 보이나요? 어느 사암이 가장 "이방성(anisotropic, 방향별로 다른 모습)" 해 보이나요?

## 3. 등방성 검토 — 방향별 공극률 프로파일

"부피가 균일한가?"를 정량적으로 보려면 한 축 방향으로 슬랩(slab)을 나누어 각 슬랩의 공극률을 봅니다.

**가설**: 세 방향(z, y, x) 모두에서 슬랩 공극률 프로파일이 비슷해야 "등방성(isotropic)" 입니다.
본 연구의 **tri-axis aggregation (세 방향 통합)** 도 이 가정에 기반합니다.

In [ ]:
# BB에 대해 세 축 프로파일 비교
vol = domains['BB']
n_slabs = 16

prof_z = porosity_profile(vol, axis=0, n_slabs=n_slabs)
prof_y = porosity_profile(vol, axis=1, n_slabs=n_slabs)
prof_x = porosity_profile(vol, axis=2, n_slabs=n_slabs)

fig, ax = plt.subplots(figsize=(9, 4))
x_axis = np.arange(n_slabs)
ax.plot(x_axis, prof_z, marker='o', label='z-axis', color=ORANGE, lw=2)
ax.plot(x_axis, prof_y, marker='s', label='y-axis', color=NAVY, lw=2)
ax.plot(x_axis, prof_x, marker='^', label='x-axis', color=GREEN, lw=2)
ax.axhline(porosity(vol), ls='--', color=GRAY, label=f'overall φ={porosity(vol):.3f}')
ax.set_xlabel(f'Slab index (n_slabs={n_slabs})')
ax.set_ylabel('Porosity φ')
ax.set_title('BB sandstone — porosity profile along three axes')
ax.legend()
plt.tight_layout()
plt.show()

# 세 축의 분산 (등방성 지표 — 작을수록 등방성)
print(f'z-axis std:  {prof_z.std():.5f}')
print(f'y-axis std:  {prof_y.std():.5f}')
print(f'x-axis std:  {prof_x.std():.5f}')

> **[Try-it! ②]** 위 셀의 `vol = domains['BB']` 를 `'CastleGate'`, `'Bentheimer'` 로 바꿔보고 결과를 비교하세요.
> 또 `n_slabs` 를 4, 16, 64로 바꾸면 곡선이 어떻게 변하는지 관찰하세요.
>
> **[해석 질문 3]** 어느 도메인이 가장 "등방성"이 좋아 보이나요? std가 가장 작은 축은 어디인가요?
> n_slabs 가 너무 작으면(예: 4) 정보가 손실되고, 너무 크면(예: 64) 노이즈가 커집니다. 적절한 trade-off는?

## 4. Sparse Imaging — 우리 연구의 문제 정의

**문제 상황**: micro-CT 스캔은 시간과 비용이 비쌉니다. 모든 슬라이스를 측정하면 시간 100%.

**Sparse 전략**: z축으로 `k` 슬라이스마다 1개만 측정 → 시간 `(1 − 1/k) × 100%` 절감.
- k=1 → 시간 절감 0% (전체 측정)
- k=3 → 약 67% 절감
- k=5 → 80% 절감

**핵심 질문**: 누락된 슬라이스를 "측정된 슬라이스로부터 얼마나 정확히 복원할 수 있는가?"

In [ ]:
vol = domains['BB']
k = 3   # ← 이 값을 바꿔보세요
axis = 0

known_idx, missing_idx = make_sparse(vol, k=k, axis=axis)
print(f'k={k}, axis={axis}: 측정 {len(known_idx)}장 / 누락 {len(missing_idx)}장 / 시간절감 {(1-1/k)*100:.1f}%')

# 연속 6 슬라이스에서 어느 것이 측정되고 누락되는지
fig, axes = plt.subplots(1, 6, figsize=(14, 3))
for i, z in enumerate(range(60, 66)):
    axes[i].imshow(vol[z, :, :])
    is_known = z in known_idx
    status = 'measured' if is_known else 'MISSING'
    color = GREEN if is_known else RED
    axes[i].set_title(f'z={z}\n{status}', color=color, fontsize=10)
    axes[i].axis('off')
plt.suptitle(f'Sparse k={k}: 연속 6장 중 어떤 슬라이스가 측정되는가', y=1.05)
plt.tight_layout()
plt.show()

> **[Try-it! ③]** 위 셀의 `k` 를 1, 2, 3, 5, 7, 10 으로 바꿔가며 측정/누락 슬라이스 수와 시간 절감률이 어떻게 변하는지 관찰하세요.
>
> **[해석 질문 4]** 시간 절감률을 늘릴수록(=k를 키울수록) 무엇이 어려워질 것이라 예상되나요? 한 줄로 답해보세요.

## 5. 가장 단순한 복원 — 선형 보간 (Linear Interpolation)

두 측정된 슬라이스 사이를 "가운데를 적당히 섞어서" 만드는 방법입니다.

$$\text{predicted}(z) = (1 - \alpha) \cdot \text{slice}_{\text{before}} + \alpha \cdot \text{slice}_{\text{after}}$$

이것이 본 연구의 **Baseline B1 (Linear)** 의 핵심 아이디어입니다.
W2에서는 scipy를 이용한 정식 구현을, W3 이후 deep learning 버전을 다룹니다.

In [ ]:
# 한 짝의 슬라이스를 선형 보간 — alpha 변화에 따른 결과
vol = domains['BB']
z_before, z_after = 60, 66

alphas = [0.0, 0.25, 0.5, 0.75, 1.0]
fig, axes = plt.subplots(1, len(alphas), figsize=(14, 3))
for i, a in enumerate(alphas):
    pred = linear_interpolate_slice(vol[z_before], vol[z_after], a)
    axes[i].imshow(pred, vmin=0, vmax=1)
    axes[i].set_title(f'α={a:.2f}')
    axes[i].axis('off')
plt.suptitle(f'두 슬라이스 사이의 선형 보간 (z={z_before} ↔ z={z_after})', y=1.05)
plt.tight_layout()
plt.show()

> **[Try-it! ④]** `z_before, z_after` 의 간격을 1 (인접) → 5 → 15 → 40 으로 늘려보세요.
> 간격이 클수록 α=0.5 결과가 어떻게 변하나요? (힌트: 흐릿함, 비현실적 픽셀)
>
> **[해석 질문 5]** 보간 결과가 0~1 사이의 "실수" 입니다. 원본은 0/1 binary 였는데요. 왜 그럴까요? 실제로 "복원된 부피" 로 쓰려면 어떤 후처리가 필요할까요?

## 6. 미니 ML 데모 — 전체 부피 sparse → 복원 → 평가

이제 한 슬라이스가 아니라 **전체 부피 256³ 를 sparse 시뮬레이션 → 선형 보간 복원 → 공극률 오차 측정** 까지 한 번에 해봅시다.

이 흐름이 본 연구 전체의 "훈련 → 추론 → 평가" 파이프라인과 정확히 같은 구조입니다 (deep learning만 빠짐).

**핵심 metric**: $|\Delta\phi| = |\phi(\text{복원}) - \phi(\text{원본})|$ — 작을수록 좋음.

In [ ]:
vol = domains['BB']
k = 5   # ← 이 값을 바꿔보세요

recon = reconstruct_sparse_linear(vol, k=k, axis=0)
err = porosity_error(recon, vol)

print(f'k={k} 선형 보간 복원')
print(f'  원본 공극률:    {porosity(vol):.4f}')
print(f'  복원 공극률:    {porosity(recon):.4f}')
print(f'  |Δφ| (오차):    {err:.4f}  ({err*100:.2f}%p)')

# 시각화: 원본 vs 복원 vs 차이
z_show = 64
fig, axes = plt.subplots(1, 3, figsize=(11, 4))
axes[0].imshow(vol[z_show]); axes[0].set_title(f'원본 z={z_show}'); axes[0].axis('off')
axes[1].imshow(recon[z_show]); axes[1].set_title(f'복원 z={z_show}'); axes[1].axis('off')
diff = np.abs(vol[z_show].astype(float) - recon[z_show])
axes[2].imshow(diff, cmap='hot'); axes[2].set_title(f'|원본 − 복원|'); axes[2].axis('off')
plt.tight_layout()
plt.show()

## 7. k Sweep — 본 연구의 동기 정량 검증

마지막으로, **k를 sweep하면서 복원 오차가 어떻게 변하는지** 측정합니다.
이 곡선이 본 연구가 "왜 deep learning이 필요한가"를 보여주는 핵심 motivation 입니다.

In [ ]:
vol = domains['BB']
k_list = [1, 2, 3, 5, 7, 10]
errors = []
time_saving = []

for k in k_list:
    recon = reconstruct_sparse_linear(vol, k=k, axis=0)
    err = porosity_error(recon, vol)
    errors.append(err)
    time_saving.append((1 - 1/k) * 100)
    print(f'  k={k:2d}  |Δφ|={err:.4f}  ({err*100:.2f}%p)  시간절감 {(1-1/k)*100:.0f}%')

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(k_list, errors, marker='o', color=ORANGE, lw=2.5, ms=10, label='|Δφ| (porosity error)')
ax1.set_xlabel('k (sparse 간격)')
ax1.set_ylabel('|Δφ|', color=ORANGE)
ax1.tick_params(axis='y', labelcolor=ORANGE)
ax1.grid(alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(k_list, time_saving, marker='s', color=NAVY, lw=2, ls='--', label='시간 절감 (%)')
ax2.set_ylabel('시간 절감 (%)', color=NAVY)
ax2.tick_params(axis='y', labelcolor=NAVY)
ax2.spines['top'].set_visible(False)

plt.title('Linear baseline: 시간 절감 vs 복원 오차 (BB sandstone)')
fig.tight_layout()
plt.show()

> **[Try-it! ⑤]** 위 셀의 `vol = domains['BB']` 를 다른 도메인으로 바꿔서 같은 분석을 반복하세요.
> 도메인 간 차이가 있나요?
>
> **[해석 질문 6]** k가 증가할수록 오차도 증가합니다. 이 곡선의 "기울기" 가 의미하는 것은 무엇인가요?
> Deep learning 모델이 이 곡선을 어떻게 바꾼다면 "실용적 진보" 라 부를 수 있을까요?

## 8. 자기 점검 & 다음 주 예고

### 자기 점검 질문
1. Voxel과 pixel의 차이는?
2. 본 데이터의 "1"은 무엇을 의미하는가?
3. 공극률을 "slab 별로 분해" 했을 때 std가 작다는 것은 어떤 의미인가?
4. k=5 sparse일 때 시간 절감률은? 그때 BB 사암의 선형 보간 |Δφ| 는 대략 얼마였나?
5. 세 축 슬라이스 패턴이 비슷한 것은 본 연구의 어떤 가정과 직결되는가?

### 다음 주 (W2) 예고
- 본격 baseline 구현: **scipy 기반 선형/Cubic interpolation**
- 한 baseline당 평가지표 다중화 — Δφ 뿐 아니라 표면적, 구조 유사도(SSIM)까지
- W3 (UNet 학습) 진입 전 마지막 "non-DL" 주차

---

## 🎯 W1 탐구 과제 (1조 — 똑똑한 너희를 위한)

**과제 1 (필수)**: 세 도메인 모두에 대해 k sweep을 수행하고, 결과를 한 plot에 겹쳐 그리세요. 어느 도메인의 sparse 보간이 가장 "쉬운가"? 그 이유를 가설로 제시.

**과제 2 (필수)**: `make_sparse(vol, k=3, axis=1)` 처럼 axis를 바꿔서 sparse 시뮬레이션도 가능합니다. **z, y, x 세 축 각각에 대해 k=3 보간 오차를 측정**하고 비교하세요. 세 축의 오차가 다르다면, 그 차이는 무엇을 의미하나요?

**과제 3 (선택 — 도전)**: `reconstruct_sparse_linear` 안에서 `(interp > 0.5)` 로 이진화하는 부분이 있습니다. 이 임계값을 0.3, 0.5, 0.7 로 바꿔보면 복원 결과의 공극률이 어떻게 변할까요? 직접 함수를 복사·수정해서 실험하고, "임계값과 공극률의 monotonic 관계" 를 표로 정리.

**과제 4 (선택 — 심화)**: 본 노트북은 "k 슬라이스마다 1개" 라는 결정적(deterministic) sparse 시나리오를 다뤘습니다. 만약 **random sparse** (전체 슬라이스에서 무작위로 30%만 측정) 라면 어떻게 시뮬레이션할 수 있을까요? `numpy.random.choice` 를 활용해 직접 구현해보세요. 결정적 sparse와 결과가 어떻게 다른가요?